# 2. Experiments and decisions

One run is one sample. To decide anything we need replications, confidence intervals and
fair comparisons. This notebook uses the built-in M/M/c queue.

In [ ]:
from simulsi import Experiment, Scenario, compare
from simulsi.analysis import select_best
from simulsi.models import mmc

model = mmc.with_options(duration=2_000, warmup=200)

Three staffing options, 10 replications each, sharing random numbers (common random numbers).

In [ ]:
scenarios = [Scenario(f"servers={c}", {"servers": c, "arrival_rate": 1.6}) for c in (2, 3, 4)]
result = Experiment(model, scenarios, replications=10, seed=42).run()
result

Differences from the 2-server baseline, with Holm-adjusted p-values for the several comparisons:

In [ ]:
compare(
    result,
    "servers=2",
    metrics=["resource.server.wait.mean", "resource.server.utilization"],
    adjust="holm",
)

## Enough replications?

`run_until` keeps adding replications until the interval is within 5% of the mean.

In [ ]:
precise = Experiment(model, scenarios[1], replications=5, seed=7).run_until(
    0.05, ["resource.server.wait.mean"], max_replications=200
)
print(precise.metadata.stopping)

## Pick the best, with a guarantee

`select_best` (procedure KN) adds replications only where needed and stops when one option is best
with 95% confidence, treating differences below `indifference` as ties.

In [ ]:
choice = select_best(
    model, {s.name: s.parameters for s in scenarios}, "resource.server.wait.mean", indifference=0.05
)
choice

## Share the result

In [ ]:
html = result.report(
    "report.html", metrics=["resource.server.wait.mean", "resource.server.utilization"]
)
print("wrote report.html,", len(html), "characters")